# Ministral 8B — Fine-tuning QLoRA pour TravelOrderResolver

**Instructions** : exécuter les cellules dans l'ordre, une par une.
Ne PAS redémarrer la session. Ne PAS rafraîchir la page.

## 1. Installation

In [ ]:
# Installer trl EN PREMIER (requis par unsloth)
!pip install trl peft accelerate bitsandbytes datasets xformers
print("✅ Dépendances de base installées")

In [ ]:
# Installer unsloth APRES les dépendances
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "unsloth_zoo @ git+https://github.com/unslothai/unsloth-zoo.git"
print("✅ Unsloth installé")

## 2. Charger le modèle

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    "unsloth/Ministral-8B-Instruct-2410-bnb-4bit",
    max_seq_length=512,
    load_in_4bit=True,
    dtype=None,
)

print(f"✅ Modèle chargé sur {model.device}")

## 3. Appliquer LoRA

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    lora_alpha=64,
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",
)

model.print_trainable_parameters()

## 4. Uploader les données

Uploadez `train.json` et `val.json` depuis `datasets/base/` du projet.

In [ ]:
from google.colab import files

print("Uploadez train.json et val.json")
uploaded = files.upload()

## 5. Préparer le dataset

In [ ]:
import json
from datasets import Dataset

UNIFIED_PROMPT = """[INST] Analyse cette phrase de voyage.
Reponds en JSON avec: langue (fr/en/other), intention (TRIP/NOT_TRIP/UNKNOWN),
et si TRIP: depart, destination, intermediaires.

Phrase: {text} [/INST]"""

def format_example(example):
    text = example.get("sentence", "")
    intent = example.get("intent", "UNKNOWN")
    lang = example.get("language", "fr")

    response = {"langue": lang, "intention": intent}

    if intent == "TRIP":
        response["depart"] = example.get("departure", None)
        response["destination"] = example.get("destination", None)
        intermediates = example.get("intermediate", [])
        if isinstance(intermediates, str):
            intermediates = [s.strip() for s in intermediates.split(",") if s.strip()]
        response["intermediaires"] = intermediates
    else:
        response["depart"] = None
        response["destination"] = None
        response["intermediaires"] = []

    prompt = UNIFIED_PROMPT.format(text=text)
    full_text = prompt + " " + json.dumps(response, ensure_ascii=False) + tokenizer.eos_token
    return {"text": full_text}

with open("train.json", "r", encoding="utf-8") as f:
    train_data = json.load(f)
with open("val.json", "r", encoding="utf-8") as f:
    val_data = json.load(f)

train_dataset = Dataset.from_list([format_example(ex) for ex in train_data])
val_dataset = Dataset.from_list([format_example(ex) for ex in val_data])

print(f"✅ Train: {len(train_dataset)} | Val: {len(val_dataset)}")
print(f"\nExemple:\n{train_dataset[0]['text'][:400]}")

## 6. Entraînement

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    max_seq_length=512,
    args=TrainingArguments(
        output_dir="ministral-unified-lora",
        num_train_epochs=3,
        per_device_train_batch_size=2,
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=1e-4,
        weight_decay=0.001,
        warmup_ratio=0.03,
        lr_scheduler_type="cosine",
        fp16=True,
        logging_steps=25,
        eval_strategy="steps",
        eval_steps=250,
        save_steps=250,
        save_total_limit=3,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        report_to="none",
        optim="adamw_8bit",
    ),
)

print("✅ Trainer prêt. Lancement...")
trainer.train()

## 7. Tester le modèle

In [ ]:
FastLanguageModel.for_inference(model)

def test_inference(text):
    prompt = UNIFIED_PROMPT.format(text=text)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=256, temperature=0.1, do_sample=False)
    generated = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)

for text in [
    "Je voudrais aller de Paris a Lyon",
    "Je veux aller de Marseille a Nice en passant par Toulon",
    "Quel temps fait-il demain?",
    "I want to go from Paris to Berlin",
]:
    print(f"\nInput: {text}")
    print(f"Output: {test_inference(text)}")
    print("-" * 40)

## 8. Sauvegarder et télécharger les poids LoRA

In [ ]:
model.save_pretrained("ministral-unified-lora")
tokenizer.save_pretrained("ministral-unified-lora")
print("✅ Poids sauvegardés")

!zip -r ministral-unified-lora.zip ministral-unified-lora/
print("✅ Archive créée")

from google.colab import files
files.download("ministral-unified-lora.zip")
print("📥 Téléchargement lancé")
print("Place le contenu du zip dans models/ministral-unified-lora/ du projet.")